# 01 · Text extraction (PDF + DOCX)

**Goal:** verify that every file in `corpus/` is validated and extracted correctly *before* chunking.
The logic lives in `app/ingestion/` (so the API reuses it); this notebook exercises it step by step.

Checks: file validation, per-document stats, a text sample, edge cases (scanned / empty / corrupt), and a PDF table experiment.

In [ ]:
import sys, pathlib, statistics
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))

from app.config import get_settings
from app.errors import AppError
from app.ingestion.validate import validate_upload
from app.ingestion.extract import extract_text

settings = get_settings()
MAX_BYTES = settings.max_upload_mb * 1024 * 1024
CORPUS = ROOT / "corpus"
files = sorted(p for p in CORPUS.iterdir() if p.suffix.lower() in {".pdf", ".docx"})
print(f"{len(files)} supported file(s) in {CORPUS}")

## 1. Validate + extract every corpus file

In [ ]:
docs = {}
print(f"{'file':40} {'type':5} {'units':>5} {'chars':>8}  sha256")
for path in files:
    data = path.read_bytes()
    try:
        upload = validate_upload(path.name, data, MAX_BYTES)
        doc = extract_text(upload.file_type, data, max_pages=settings.max_pdf_pages)
    except AppError as exc:
        print(f"{path.name:40} REJECTED ({exc.status_code}): {exc.message}")
        continue
    docs[path.name] = doc
    print(f"{path.name[:40]:40} {doc.file_type:5} {len(doc.sections):>5} {doc.total_chars:>8}  {upload.sha256[:12]}")

## 2. Chars per page / section
Very small values often mean headers-only pages, slides or scanned pages.

In [ ]:
for name, doc in docs.items():
    sizes = [len(s.text) for s in doc.sections]
    print(f"{name[:40]:40} min={min(sizes):>6} median={int(statistics.median(sizes)):>6} max={max(sizes):>6}")

## 3. Eyeball the text
Check reading order, broken words, repeated headers/footers, garbled characters.

In [ ]:
for name, doc in docs.items():
    print("=" * 80, f"\n{name}")
    for s in doc.sections[:2]:
        where = f"page {s.page}" if s.page else f"section '{s.section}'"
        print(f"--- {where} ---\n{s.text[:700]}\n")

**My observations (fill in, reused in DESIGN.md):**
- Reading order: ...
- Repeated headers/footers: ...
- Tables: ...
- Special characters / non-English: ...

## 4. Edge cases: each must fail cleanly with the right status

In [ ]:
import io, pymupdf
from docx import Document

def blank_pdf():
    pdf = pymupdf.open(); page = pdf.new_page()
    page.draw_rect(pymupdf.Rect(50, 50, 200, 200)); return pdf.tobytes()

def empty_docx():
    buf = io.BytesIO(); d = Document(); d.add_paragraph(" "); d.save(buf); return buf.getvalue()

cases = {
    "scanned/blank PDF": ("scan.pdf", blank_pdf()),
    "empty DOCX": ("empty.docx", empty_docx()),
    "zero-byte file": ("zero.pdf", b""),
    "renamed .exe": ("fake.pdf", b"MZ\x90\x00 not a pdf"),
    "unsupported type": ("notes.xyz", b"hello"),
    "truncated PDF": ("bad.pdf", b"%PDF-1.4 garbage"),
}
for label, (fname, blob) in cases.items():
    try:
        up = validate_upload(fname, blob, MAX_BYTES)
        extract_text(up.file_type, blob)
        print(f"{label:20} -> accepted (unexpected!)")
    except AppError as exc:
        print(f"{label:20} -> {exc.status_code} {exc.code}: {exc.message}")

## 5. PDF table experiment
Plain text extraction can scramble tables. Compare with PyMuPDF's table finder on your first PDF that has tables.

In [ ]:
pdfs = [p for p in files if p.suffix.lower() == ".pdf"]
found = False
for path in pdfs:
    with pymupdf.open(path) as pdf:
        for number, page in enumerate(pdf, start=1):
            tables = page.find_tables().tables
            if tables:
                found = True
                print(f"{path.name} page {number}: {len(tables)} table(s)")
                print(tables[0].to_markdown()[:800])
                print("--- plain-text extraction of the same page ---")
                print(page.get_text('text', sort=True)[:800])
                break
    if found: break
if not found:
    print("No tables detected in corpus PDFs.")

**Decision to record in DESIGN.md:** if `find_tables` output is clearly better than plain text for your corpus, switch the PDF extractor to serialise tables like the DOCX one (`Header: value | ...`). Otherwise keep plain text and note the limitation in EVAL.md.